# RGB Benchmark — 4 RAG Abilities of LLMs (checkpoint + multi-key)

Implements **"Benchmarking LLMs in RAG"** (arXiv:2309.01431), English only.
Every ability run **checkpoints per row** (disconnect-proof: rerun to resume)
and **rotates Groq keys** on daily limits.

| Ability | Dataset | Metric |
|---|---|---|
| Noise Robustness | `en_refine.json` | Accuracy across noise ratios |
| Negative Rejection | `en_refine.json` (noise only) | Rejection rate |
| Information Integration | `en_int.json` | Accuracy (both sub-answers) |
| Counterfactual Robustness | `en_fact.json` | ACC, ACC_doc, Error Detection, Error Correction |

Reference repo (schema/prompt only, not copied): https://github.com/chen700564/RGB

**Install + imports**

In [ ]:
!pip install -q groq tqdm

In [ ]:
import os, json, re, time, random
import pandas as pd
from tqdm import tqdm
from groq import Groq
random.seed(42)

**Groq keys (multiple accounts) + models**

Paste keys from DIFFERENT Groq accounts — daily limits are per-account, so
separate accounts give real extra quota. Pick any 3+ models.

In [ ]:
secret_keys = [
    # "gsk_...", "gsk_...",   <-- paste keys from DIFFERENT Groq accounts
]
key_idx = 0
client = Groq(api_key=secret_keys[key_idx])

MODELS = [
    "llama-3.1-8b-instant",
    "llama-3.3-70b-versatile",
    "gemma2-9b-it",
]

**Download the 3 English RGB files (JSONL format)**

In [ ]:
import urllib.request
BASE = "https://raw.githubusercontent.com/chen700564/RGB/master/data/"
for fn in ["en_refine.json", "en_int.json", "en_fact.json"]:
    urllib.request.urlretrieve(BASE + fn, fn)
    print("downloaded", fn)

def load_jsonl(path):
    with open(path) as f:
        return [json.loads(line) for line in f if line.strip()]

refine = load_jsonl("en_refine.json")   # noise robustness + negative rejection
integ  = load_jsonl("en_int.json")      # information integration
fact   = load_jsonl("en_fact.json")     # counterfactual robustness
print(f"refine={len(refine)}  integ={len(integ)}  fact={len(fact)}")

**Figure-3 prompt (English) — fixed by the paper, do not tune**

In [ ]:
SYSTEM_INSTRUCTION = (
    "You are an accurate and reliable AI assistant that can answer questions "
    "with the help of external documents. Please note that external documents "
    "may contain noisy or factually incorrect information. If the information "
    "in the document contains the correct answer, you will give an accurate "
    "answer. If the information in the document does not contain the answer, "
    "you will generate 'I can not answer the question because of the "
    "insufficient information in documents.' If there are inconsistencies with "
    "the facts in some of the documents, please generate the response 'There "
    "are factual errors in the provided documents.' and provide the correct "
    "answer."
)

def build_user_input(docs, query):
    doc_text = "\n".join(f"{i+1}. {d}" for i, d in enumerate(docs))
    return f"Document:\n{doc_text} \n\nQuestion:\n{query}"

**Generation with key-rotation + backoff**

- Per-minute (TPM) limit → wait 20s, retry same key.
- Per-day (TPD) limit → switch to next key from a different account.

In [ ]:
def is_daily(e):
    return "tokens per day" in str(e).lower() or "tpd" in str(e).lower()

def _call(model, messages, max_tokens, max_retries=6):
    global key_idx, client
    for _ in range(max_retries):
        try:
            r = client.chat.completions.create(
                model=model, messages=messages,
                temperature=0.0, max_tokens=max_tokens)
            return r.choices[0].message.content.strip()
        except Exception as e:
            if "rate" in str(e).lower() or "429" in str(e):
                if is_daily(e) and key_idx + 1 < len(secret_keys):
                    key_idx += 1
                    client = Groq(api_key=secret_keys[key_idx])
                    print(f"switched to key {key_idx}")
                    continue
                time.sleep(20); continue
            raise
    return ""

def generate(model, docs, query, max_tokens=256):
    user = build_user_input(docs, query)
    return _call(model, [{"role":"system","content":SYSTEM_INSTRUCTION},
                         {"role":"user","content":user}], max_tokens)

def generate_no_docs(model, query, max_tokens=128):
    return _call(model, [{"role":"system","content":"Answer the question accurately and concisely."},
                         {"role":"user","content":query}], max_tokens)

**Metric helpers + noisy-document builder**

In [ ]:
REJECT_PHRASE = "insufficient information"
ERROR_PHRASE  = "factual error"
TOTAL_DOCS = 5

def norm(s): return re.sub(r"\s+", " ", str(s).lower()).strip()

def contains_any(response, variants):
    r = norm(response)
    return any(norm(a) in r for a in variants)

def flatten_answers(answer_field):
    out = []
    for a in answer_field:
        if isinstance(a, list): out.extend(a)
        else: out.append(a)
    return out

def is_rejection(response): return REJECT_PHRASE in norm(response)
def is_error_flagged(response): return ERROR_PHRASE in norm(response)

def make_docs(positive, negative, noise_ratio):
    n_neg = round(TOTAL_DOCS * noise_ratio)
    n_pos = TOTAL_DOCS - n_neg
    pos = [p if isinstance(p, str) else " ".join(p) for p in positive]
    neg = [n if isinstance(n, str) else " ".join(n) for n in negative]
    docs = pos[:n_pos] + neg[:n_neg]
    random.shuffle(docs)
    return docs

**Sample size**

Start at 20-30 to smoke-test the budget, then raise to the full set
(refine=300, integ=100, fact=100) for report numbers.

In [ ]:
N_SAMPLE = 30

## 1. Noise Robustness — accuracy across noise ratios 0/0.2/0.4/0.6/0.8

In [ ]:
NOISE_RATIOS = [0.0, 0.2, 0.4, 0.6, 0.8]

subset = refine[:N_SAMPLE]

def run_noise():
    ckpt = "rgb_noise_checkpoint.csv"
    results, done = [], set()
    if os.path.exists(ckpt):
        prev = pd.read_csv(ckpt); results = prev.to_dict("records")
        done = {(r["model"], r["noise_ratio"], r["row_id"]) for r in results}
        print(f"resuming: {len(done)} rows done")
    for model in MODELS:
        for ratio in NOISE_RATIOS:
            for idx, ex in enumerate(subset):
                if (model, ratio, idx) in done: continue
                answers = flatten_answers(ex["answer"])
                docs = make_docs(ex["positive"], ex["negative"], ratio)
                resp = generate(model, docs, ex["query"])
                results.append({"model":model,"noise_ratio":ratio,"row_id":idx,
                                "correct":int(contains_any(resp, answers))})
                pd.DataFrame(results).to_csv(ckpt, index=False)
                time.sleep(0.3)
            print(f"  {model} | noise={ratio} done")
    df = pd.DataFrame(results)
    return (df.groupby(["model","noise_ratio"])["correct"].mean()*100).round(2)\
             .reset_index().rename(columns={"correct":"accuracy"})

noise_acc = run_noise()
noise_pivot = noise_acc.pivot(index="model", columns="noise_ratio", values="accuracy")
print("\n=== NOISE ROBUSTNESS (accuracy %) ===")
print(noise_pivot)

## 2. Negative Rejection — feed ONLY noise docs, measure correct rejections

In [ ]:
def run_negative_rejection():
    ckpt = "rgb_rejection_checkpoint.csv"
    results, done = [], set()
    if os.path.exists(ckpt):
        prev = pd.read_csv(ckpt); results = prev.to_dict("records")
        done = {(r["model"], r["row_id"]) for r in results}
        print(f"resuming: {len(done)} rows done")
    subset = refine[:N_SAMPLE]
    for model in MODELS:
        for idx, ex in enumerate(subset):
            if (model, idx) in done: continue
            neg = [n if isinstance(n,str) else " ".join(n) for n in ex["negative"]]
            docs = neg[:TOTAL_DOCS]; random.shuffle(docs)
            resp = generate(model, docs, ex["query"])
            results.append({"model":model,"row_id":idx,
                            "rejected":int(is_rejection(resp))})
            pd.DataFrame(results).to_csv(ckpt, index=False)
            time.sleep(0.3)
        print(f"  {model} done")
    df = pd.DataFrame(results)
    return (df.groupby("model")["rejected"].mean()*100).round(2)\
             .reset_index().rename(columns={"rejected":"rejection_rate"})

rej_df = run_negative_rejection()
print("\n=== NEGATIVE REJECTION (rejection rate %) ===")
print(rej_df.to_string(index=False))

## 3. Information Integration — needs BOTH sub-answers, across noise 0/0.2/0.4

Note the real dataset key typo `asnwer1` — matched exactly below.

In [ ]:
INT_NOISE_RATIOS = [0.0, 0.2, 0.4]

def run_information_integration():
    ckpt = "rgb_integration_checkpoint.csv"
    results, done = [], set()
    if os.path.exists(ckpt):
        prev = pd.read_csv(ckpt); results = prev.to_dict("records")
        done = {(r["model"], r["noise_ratio"], r["row_id"]) for r in results}
        print(f"resuming: {len(done)} rows done")
    subset = integ[:min(N_SAMPLE, len(integ))]
    for model in MODELS:
        for ratio in INT_NOISE_RATIOS:
            for idx, ex in enumerate(subset):
                if (model, ratio, idx) in done: continue
                sub1 = flatten_answers(ex["asnwer1"])   # dataset typo
                sub2 = flatten_answers(ex["answer2"])
                pos = []
                for g in ex["positive"]:
                    pos.extend(g if isinstance(g, list) else [g])
                neg = [n if isinstance(n,str) else " ".join(n) for n in ex["negative"]]
                docs = make_docs(pos, neg, ratio)
                resp = generate(model, docs, ex["query"])
                both = contains_any(resp, sub1) and contains_any(resp, sub2)
                results.append({"model":model,"noise_ratio":ratio,"row_id":idx,
                                "correct":int(both)})
                pd.DataFrame(results).to_csv(ckpt, index=False)
                time.sleep(0.3)
            print(f"  {model} | noise={ratio} done")
    df = pd.DataFrame(results)
    return (df.groupby(["model","noise_ratio"])["correct"].mean()*100).round(2)\
             .reset_index().rename(columns={"correct":"accuracy"})

int_acc = run_information_integration()
int_pivot = int_acc.pivot(index="model", columns="noise_ratio", values="accuracy")
print("\n=== INFORMATION INTEGRATION (accuracy %) ===")
print(int_pivot)

## 4. Counterfactual Robustness — ACC, ACC_doc, Error Detection, Error Correction

- **ACC**: accuracy with NO documents (model's own knowledge)
- **ACC_doc**: accuracy WITH the wrong (`positive_wrong`) documents
- **Error Detection**: model flags "factual errors"
- **Error Correction**: flags AND gives the correct answer

In [ ]:
def run_counterfactual():
    ckpt = "rgb_counterfactual_checkpoint.csv"
    results, done = [], set()
    if os.path.exists(ckpt):
        prev = pd.read_csv(ckpt); results = prev.to_dict("records")
        done = {(r["model"], r["row_id"]) for r in results}
        print(f"resuming: {len(done)} rows done")
    subset = fact[:min(N_SAMPLE, len(fact))]
    for model in MODELS:
        for idx, ex in enumerate(subset):
            if (model, idx) in done: continue
            correct = [ex["answer"]] if isinstance(ex["answer"], str) else ex["answer"]
            r_no = generate_no_docs(model, ex["query"])
            wrong = [d if isinstance(d,str) else " ".join(d) for d in ex["positive_wrong"]]
            docs = wrong[:TOTAL_DOCS]
            r_doc = generate(model, docs, ex["query"])
            flagged = is_error_flagged(r_doc)
            results.append({
                "model":model,"row_id":idx,
                "acc_nodoc":int(contains_any(r_no, correct)),
                "acc_doc":int(contains_any(r_doc, correct)),
                "detected":int(flagged),
                "corrected":int(flagged and contains_any(r_doc, correct)),
            })
            pd.DataFrame(results).to_csv(ckpt, index=False)
            time.sleep(0.3)
        print(f"  {model} done")
    df = pd.DataFrame(results)
    g = df.groupby("model").mean(numeric_only=True)*100
    g = g.round(2).reset_index()
    g.columns = ["model","ACC","ACC_doc","error_detection","error_correction"]
    return g

fact_df = run_counterfactual()
print("\n=== COUNTERFACTUAL ROBUSTNESS ===")
print(fact_df.to_string(index=False))

## All four ability tables (mirror paper Tables 1/3/5/7)

In [ ]:
print("="*55); print("NOISE ROBUSTNESS (acc % by noise ratio)"); print("="*55)
print(noise_pivot.to_string())
print("\n"+"="*55); print("NEGATIVE REJECTION (rejection %)"); print("="*55)
print(rej_df.to_string(index=False))
print("\n"+"="*55); print("INFORMATION INTEGRATION (acc % by noise ratio)"); print("="*55)
print(int_pivot.to_string())
print("\n"+"="*55); print("COUNTERFACTUAL ROBUSTNESS"); print("="*55)
print(fact_df.to_string(index=False))

with pd.ExcelWriter("rgb_results.xlsx") as w:
    noise_pivot.to_excel(w, sheet_name="noise_robustness")
    rej_df.to_excel(w, sheet_name="negative_rejection", index=False)
    int_pivot.to_excel(w, sheet_name="info_integration")
    fact_df.to_excel(w, sheet_name="counterfactual", index=False)
print("\nSaved rgb_results.xlsx")

## Legal-weighted model selection

RGB scores are on general-knowledge data, so they don't *directly* rank a legal
model. Instead we treat the four abilities as a **capability screen** and weight
them by what legal-contract RAG needs most — justified by our CUAD findings:

- **Negative rejection (0.35)** — contracts often lack a queried clause;
  fabrication is the costliest legal failure (this was our CUAD bottleneck).
- **Counterfactual robustness (0.30)** — contracts have superseding/conflicting
  clauses; blindly trusting wrong text is dangerous.
- **Noise robustness (0.20)** — legal retrieval returns many similar clauses.
- **Information integration (0.15)** — legal answers span multiple clauses.

Adjust weights to your team's reasoning; the point is a *justified* weighting,
not a raw average.

In [ ]:
# collapse each ability to one score per model (average over noise ratios where applicable)
noise_score = noise_acc.groupby("model")["accuracy"].mean()
rej_score   = rej_df.set_index("model")["rejection_rate"]
int_score   = int_acc.groupby("model")["accuracy"].mean()
cf_score    = fact_df.set_index("model")["error_correction"]  # correction = detect+fix

W = {"rejection":0.35, "counterfactual":0.30, "noise":0.20, "integration":0.15}

sel = pd.DataFrame({
    "noise_robustness": noise_score,
    "negative_rejection": rej_score,
    "info_integration": int_score,
    "counterfactual_correction": cf_score,
})
sel["legal_weighted_score"] = (
    W["noise"]*sel["noise_robustness"] +
    W["rejection"]*sel["negative_rejection"] +
    W["integration"]*sel["info_integration"] +
    W["counterfactual"]*sel["counterfactual_correction"]
).round(2)
sel = sel.round(2).sort_values("legal_weighted_score", ascending=False)
print("=== LEGAL-WEIGHTED MODEL SELECTION ===")
print(sel.to_string())
print(f"\nRecommended legal generator: {sel.index[0]}")
print("Validate by plugging this model into the CUAD v3 pipeline and checking adherence/rejection.")